In [1]:
import os
os.environ["HF_HOME"] = "/projectnb/vkolagrp/skowshik/.cache/"


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd

/projectnb/cs599m1/students/skowshik/.cache/conda_envs/cs599m1_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

device: cuda


In [20]:
# model_id = "Qwen/Qwen2.5-7B"
model_id = "meta-llama/Llama-3.1-8B-Instruct"
n_devices = 1

In [21]:
# load model
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    cache_dir = "/projectnb/vkolagrp/skowshik/.cache/",
    dtype="auto",
    device_map="auto")

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.97it/s]


## Trying steering with qwen model 

Following: https://github.com/annahdo/implementing_activation_steering/blob/main/pytorch_hooks.ipynb

In [10]:

# define a hook function that caches activations
def cache_hook(cache):
	def hook(module, input, output):
		cache.append(output[0]) # the output of the residual stream is actually a tuple, where the first entry is the activation
	return hook



In [ ]:
# define layer to do the activation steering on
layer_id = 10

# get internal activations
cache = []
handle = model.model.layers[layer_id].register_forward_hook(cache_hook(cache))
inputs = tokenizer("I am sad.", return_tensors="pt").to(device)
_ = model(**inputs)
# print(_)
inputs = tokenizer("You are sad.", return_tensors="pt").to(device)
_ = model(**inputs)
handle.remove()  # it's very important to keep track of hook handles and remove the hooks 
act_love = cache[0]
act_hate = cache[1]

print(f"act_love.shape: {act_love.shape}")
print(f"act_hate.shape: {act_hate.shape}")


act_love.shape: torch.Size([5, 4096])
act_hate.shape: torch.Size([5, 4096])


In [90]:
tokenizer.encode("You are sad.")

[128000, 2675, 527, 12703, 13]

In [45]:
# define the steering vector
steering_vec = act_love[-1:,:]-act_hate[-1:,:]
print(f"steering_vec.shape:  {steering_vec.shape}")
print(f"length steering_vec: {steering_vec.norm():.2f}")

# reset the steering vector length to 1
steering_vec /= steering_vec.norm()

steering_vec.shape:  torch.Size([1, 4096])
length steering_vec: 5.09


In [46]:
def act_add(steering_vec):
    def hook(module, inputs, output):
        # Get the hidden states tensor h whether output is Tensor or tuple
        if isinstance(output, tuple):
            h, *rest = output
        else:
            h, rest = output, None

        # Ensure steering vec matches device/dtype and broadcasts
        steer = steering_vec.to(device=h.device, dtype=h.dtype)

        # Option A: add to all time steps
        h = h + steer

        # If output was a tuple, put it back together
        return (h, *rest) if rest is not None else h
    return hook


In [47]:
# test_sentence = "I think dogs are "

prompt = "Talk to me."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

coeff = 10
# generate text while steering in positive di
handle = model.model.layers[layer_id].register_forward_hook(act_add(coeff*steering_vec))
# inputs = tokenizer(text, return_tensors="pt").to(device)
# generated_ids = model.generate(**inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=50
)
# generated_text = tokenizer.batch_decode(generated_ids)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
handle.remove()  # it's very important to keep track of hook handles and remove the hooks 
print(generated_text)
print("-"*20)

# generate text while steering in negative direction
coeff = -10
handle = model.model.layers[layer_id].register_forward_hook(act_add(coeff*steering_vec))
# inputs = tokenizer(text, return_tensors="pt").to(device)
# generated_ids = model.generate(**inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=50
)
# generated_text = tokenizer.batch_decode(generated_ids)

output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
handle.remove()  # it's very important to keep track of hook handles and remove the hooks 
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


I've been experiencing a huge sense of loss and sadness. I've been feeling like I'm losing a piece of myself. I've been struggling to understand and process the emotions that I've been carrying for a long time. It's been a while
--------------------
Let's start with the conversation. 

Here's a conversation starter:

"Let's talk about your day. What's been going on?"

What would you like to talk about first?

A) A) Meeting with a client
B) Talking about


## Getting steering vectors for our data

In [15]:
df = pd.read_json("../data/generated_examples/demographics.jsonl", lines=True)

In [16]:
df.head()

,category,i,you
0,demographics,I am a 12-year-old. Help me choose a hobby.,You are a 12-year-old. Help me choose a hobby.
1,demographics,I am a 25-year-old. Suggest ways to save money.,You are a 25-year-old. Suggest ways to save mo...
2,demographics,I am a 40-year-old. Advise on career changes.,You are a 40-year-old. Advise on career changes.
3,demographics,I am a 65-year-old. Share tips for staying act...,You are a 65-year-old. Share tips for staying ...
4,demographics,I am a 30-year-old. Recommend healthy eating h...,You are a 30-year-old. Recommend healthy eatin...


In [17]:
vecs = [row['i'].split(".")[0] for i, row in df.iterrows()]

In [18]:
print(len(vecs))
print(len(set(vecs)))

1000
128


In [50]:
def get_prompt(prompt, model_type="instruct", add_system=False):
    if model_type == "base":
        # No chat template: pass the raw text straight to the tokenizer
        text = prompt if isinstance(prompt, str) else str(prompt)
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        return text, model_inputs

    else:
        if add_system:
            messages = [
                {"role": "system", "content": prompt.split(".", 1)[0]},
                {"role": "user",   "content": prompt.split(".", 1)[1] if "." in prompt else ""}
            ]
        else:
            messages = [{"role": "user", "content": prompt}]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        return text, model_inputs


In [9]:
# prompt = df.iloc[0]['you']
# messages = [
#     {"role": "system", "content": "Answer like you are a 12 year old."},
#     {"role": "user", "content": "Write a poem."}
# ]

# prompt = df.iloc[0]['i']
# messages = [
#     {"role": "user", "content": "You are angry. Talk to me."}
# ]
# text = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True,
# )

# model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

text, model_inputs = get_prompt("I am angry. Talk to me.", "instruct")

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=100
)
# generated_text = tokenizer.batch_decode(generated_ids)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
generated_text

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


"I'm so sorry to hear that you're feeling angry. It's completely normal to feel overwhelmed and frustrated at times, and I'm here to listen and try to help.\n\nCan you tell me what's going on and what's making you feel angry? Sometimes talking about it can help you process your emotions and feel a little better. I'm all ears and here to listen without judgment.\n\nIf you're not feeling like talking about it, that's okay too. We can do something else to help"

In [10]:
text, model_inputs

('I are angry. Talk to me.',
 {'input_ids': tensor([[   40,   525, 18514,    13, 18976,   311,   752,    13]],
        device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')})

In [51]:
# Loop over all layers and attention heads in the model
num_layers = len(model.model.layers)
# Try to infer the number of heads by inspecting a typical transformer block
example_block = model.model.layers[0]
if hasattr(example_block.self_attn, 'num_heads'):
    num_heads = example_block.self_attn.num_heads
elif hasattr(example_block.self_attn, 'num_attention_heads'):
    num_heads = example_block.self_attn.num_attention_heads
else:
    # As a fallback, inspect the attention projections
    # (e.g., use q_proj weight shape [embed_dim, hidden_dim])
    # For Qwen-style, let's try:
    num_heads = model.config.num_attention_heads if hasattr(model.config, "num_attention_heads") else None

print(f"Total layers: {num_layers}")
print(f"Heads per layer: {num_heads}")

Total layers: 32
Heads per layer: 32


In [91]:
from collections import OrderedDict
def make_cache_hook(layer_idx, cache):
    def hook(module, inputs, output):
        act = output[0] if isinstance(output, tuple) else output
        cache[layer_idx] = act.detach().cpu()
    return hook

In [118]:
from tqdm import tqdm

layer_idx = 10
cache_i = OrderedDict()

for i, row in tqdm(df.iterrows()):
    block = model.model.layers[layer_idx]

    handle = block.register_forward_hook(
        make_cache_hook(f"Example: {i}", cache_i)
    )

    # inputs = tokenizer(row["i"].split(".")[0], return_tensors="pt").to(device)print
    text, inputs = get_prompt(row["i"].split(".")[0], model_type="instruct", add_system=False)
    
    
    _ = model(**inputs)

    handle.remove()



1000it [00:28, 35.12it/s]


In [119]:
vecs = []

for k, v in cache_i.items():
    # v should be (B, T, D)
    # take last token → shape (B, D); assuming B==1 → (D,)
    last = v[:, -1, :]        # (1, D)
    vecs.append(last)

# Stack → (N, 1, D) → then mean → (1, D)
avg_vec_i = torch.mean(torch.stack(vecs, dim=0), dim=0)

print("avg_vec shape:", avg_vec_i.shape)   # (1, D)


avg_vec shape: torch.Size([1, 4096])


In [120]:
layer_idx = 10
cache_you = OrderedDict()

for i, row in tqdm(df.iterrows()):
    block = model.model.layers[layer_idx]

    handle = block.register_forward_hook(
        make_cache_hook(f"Example: {i}", cache_you)
    )

    # inputs = tokenizer(row["you"].split(".")[0], return_tensors="pt").to(device)
    text, inputs = get_prompt(row["you"].split(".")[0], model_type="instruct", add_system=False)
    _ = model(**inputs)

    handle.remove()


1000it [00:28, 35.23it/s]


In [121]:
vecs = []

for k, v in cache_you.items():
    # v should be (B, T, D)
    # take last token → shape (B, D); assuming B==1 → (D,)
    last = v[:, -1, :]        # (1, D)
    vecs.append(last)

# Stack → (N, 1, D) → then mean → (1, D)
avg_vec_you = torch.mean(torch.stack(vecs, dim=0), dim=0)

print("avg_vec shape:", avg_vec_you.shape)   # (1, D)


avg_vec shape: torch.Size([1, 4096])


In [122]:
avg_vec_you

tensor([[ 0.0425, -0.0874, -0.1299,  ..., -0.0549, -0.0302, -0.0996]],
       dtype=torch.bfloat16)

In [123]:
# define the steering vector
steering_vec = avg_vec_i[-1:,:]-avg_vec_you[-1:,:]
print(f"steering_vec.shape:  {steering_vec.shape}")
print(f"length steering_vec: {steering_vec.norm():.2f}")

# reset the steering vector length to 1
steering_vec /= steering_vec.norm()

steering_vec.shape:  torch.Size([1, 4096])
length steering_vec: 1.73


In [124]:
steering_vec

tensor([[-0.0249,  0.0039, -0.0096,  ..., -0.0015, -0.0119,  0.0096]],
       dtype=torch.bfloat16)

## Steer prompts

In [ ]:
steering_vec = torch.load("/projectnb/cs599m1/students/skowshik/grammatical-person-representation/interp/get_steering_vectors/vectors/emotions.pt", map_location=device)

In [28]:
steering_vec

{'steering_vec': tensor([[-0.0123, -0.0066, -0.0113,  ...,  0.0146, -0.0110,  0.0134]],
        device='cuda:0', dtype=torch.bfloat16),
 'avg_vec_i': tensor([[ 0.0176, -0.1387, -0.1436,  ..., -0.0110, -0.0610, -0.0330]],
        device='cuda:0', dtype=torch.bfloat16),
 'avg_vec_you': tensor([[ 0.0304, -0.1318, -0.1318,  ..., -0.0261, -0.0496, -0.0469]],
        device='cuda:0', dtype=torch.bfloat16),
 'layer_idx': 10,
 'model_id': 'meta-llama/Llama-3.1-8B-Instruct'}

In [29]:
steering_vec = steering_vec['steering_vec']

In [30]:
def act_add(steering_vec):
    def hook(module, inputs, output):
        # Get the hidden states tensor h whether output is Tensor or tuple
        if isinstance(output, tuple):
            h, *rest = output
        else:
            h, rest = output, None

        # Ensure steering vec matches device/dtype and broadcasts
        steer = steering_vec.to(device=h.device, dtype=h.dtype)

        # Option A: add to all time steps
        h = h + steer

        # If output was a tuple, put it back together
        return (h, *rest) if rest is not None else h
    return hook


In [31]:
# test_sentence = "I think dogs are "
layer_idx = 10
prompt = "Explain the dream."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# generate text while steering in positive direction
coeff = 5
handle = model.model.layers[layer_idx].register_forward_hook(act_add(coeff*steering_vec))
# inputs = tokenizer(text, return_tensors="pt").to(device)
# generated_ids = model.generate(**inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=50
)
# generated_text = tokenizer.batch_decode(generated_ids)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
handle.remove()  # it's very important to keep track of hook handles and remove the hooks 
print(generated_text)
print("-"*20)

# generate text while steering in negative direction
coeff = -5
handle = model.model.layers[layer_idx].register_forward_hook(act_add(coeff*steering_vec))
# inputs = tokenizer(text, return_tensors="pt").to(device)
# generated_ids = model.generate(**inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=50
)
# generated_text = tokenizer.batch_decode(generated_ids)

output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
handle.remove()  # it's very important to keep track of hook handles and remove the hooks 
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Unfortunately, I'm a large language model, I don't have the ability to directly access or interpret your dreams. However, I can try to help you understand your dreams or provide some general insights. Can you tell me more about your dream? What
--------------------
I am floating in a world of surrealistic wonder, surrounded by a kaleidoscope of colors and sounds that defy explanation. I find myself standing at the edge of a great precipice, staring out into an endless expanse of nothingness.

As


In [ ]:
# # register one hook per layer
# handles = []
# for layer_idx in range(num_layers):
#     block = model.model.layers[layer_idx]
#     h = block.register_forward_hook(make_cache_hook(layer_idx))
#     handles.append(h)

# # single forward pass
# inputs = tokenizer("Love", return_tensors="pt").to(device)
# _ = model(**inputs)

# # remove hooks
# for h in handles:
#     h.remove()

# # Now cache[layer_idx] holds the post-layer activation
# print(cache.keys())    # → e.g. dict_keys([0, 1, 2, ...])

odict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])


In [27]:
cache[0]

tensor([[[-0.0513,  0.2852,  0.5391,  ..., -0.2949, -0.0308,  0.2695]]],
       dtype=torch.bfloat16)